Simplified integral of Earth albedo to altitude

In [14]:
import numpy as np
import sympy as sp
phi, theta, theta_0 = sp.symbols('phi theta theta_0', real=True)
RE, h, k = sp.symbols('R_E h k', real=True)

u = sp.Matrix([
    sp.sin(theta) * sp.cos(phi),
    sp.sin(theta) * sp.sin(phi),
    sp.cos(theta)
])

u0 = sp.Matrix([
    0,
    sp.sin(theta_0),
    sp.cos(theta_0)
])

rr = (RE+h) * u0 - RE * u
rr_norm_4 = sp.simplify((rr.norm())**4)

cos_alpha = sp.simplify(rr.dot(u)/rr.norm())
cos_alpha_num = sp.simplify(rr.dot(u))
cos_alpha_num

cos_plus_theta = 1/2 * (sp.cos(theta) + sp.Abs(sp.cos(theta)))
int_1 = sp.simplify(rr.dot(u0) * rr.dot(u) * cos_plus_theta * sp.sin(theta) / rr_norm_4)
int_2 = sp.simplify((1/2) * rr.dot(u0) * (rr.dot(u) + sp.Abs(rr.dot(u))) * cos_plus_theta *sp.sin(theta) / rr_norm_4)
print(int_1)

0.5*(cos(theta) + Abs(cos(theta)))*(R_E*sin(phi)*sin(theta)*sin(theta_0) + R_E*cos(theta)*cos(theta_0) - R_E - h)*(-R_E*sin(phi)*sin(theta)*sin(theta_0) - R_E*cos(theta)*cos(theta_0) + R_E - h*sin(phi)*sin(theta)*sin(theta_0) - h*cos(theta)*cos(theta_0))*sin(theta)/(R_E**2*sin(theta)**2*cos(phi)**2 + (R_E*cos(theta) - (R_E + h)*cos(theta_0))**2 + (R_E*sin(phi)*sin(theta) - (R_E + h)*sin(theta_0))**2)**2


Integral of attenuated and non-parallel sun rays across an arbitrary control volume:

In [7]:
import numpy as np
import sympy as sp

phi, theta = sp.symbols('phi theta', real=True)
RE, h, ds, k = sp.symbols('R_E h d_s k', real=True)

u = sp.Matrix([
    sp.sin(theta) * sp.cos(phi),
    sp.sin(theta) * sp.sin(phi),
    sp.cos(theta)
])
rS = sp.Matrix([
    0,
    0,
    ds
])
rr = (RE + h) * u - rS

F_vec = sp.simplify(k * rr / (rr.norm())**3)
int = sp.simplify(F_vec.dot(u))
int

k*(R_E - d_s*cos(theta) + h)/(R_E**2 - 2*R_E*d_s*cos(theta) + 2*R_E*h + d_s**2 - 2*d_s*h*cos(theta) + h**2)**(3/2)

In [3]:
from SpaceBalls.radiation_settings import get_ae_sh_maps_numpy_new
import numpy as np

a, e = get_ae_sh_maps_numpy_new('SYN1deg_fine', '2018-01-01')
a.nbytes / 1e9 *1825 * 1440

1362.3552

In [11]:
import numpy as np
import sympy as sp

phi, lam = sp.symbols('phi lambda')
Req, e, f = sp.symbols('R_eq e f', positive=True)

#e = sp.sqrt(f*(2-f))
r = Req * (1-f) / sp.sqrt(1 - e**2*(sp.cos(lam))**2)
sp.simplify(r)

R_eq*(1 - f)/sqrt(-e**2*cos(lambda)**2 + 1)

In [ ]:
dr_dl = sp.diff(r, lam)
sp.simplify(dr_dl)

sqrt(2)*R_eq*e**2*(f - 1)*sin(2*lambda)/(-e**2*cos(2*lambda) - e**2 + 2)**(3/2)

In [13]:
sp.simplify((dr_dl / r)**2)

e**4*sin(lambda)**2*cos(lambda)**2/(e**2*cos(lambda)**2 - 1)**2

In [14]:
dS = r**2 * sp.cos(lam) * sp.sqrt(1 + (dr_dl / r)**2)
sp.simplify(dS)

-R_eq**2*sqrt((e**4*cos(lambda)**2 - 2*e**2*cos(lambda)**2 + 1)/(e**2*cos(lambda)**2 - 1)**2)*(f - 1)**2*cos(lambda)/(e**2*cos(lambda)**2 - 1)

In [15]:
# integral solved by GPT 5.6 Luna in reasoning mode

x = sp.sin(lam)
q = 1 - e**2
A = e**2 * (2 - e**2) 
integral = x * sp.sqrt(q**2 + A*x**2)/(q + e**2 * x**2) + q/(e)*sp.atanh(e * x / sp.sqrt(q**2 + A*x**2))
sp.simplify(integral)

(e*sqrt(-e**2*(e**2 - 2)*sin(lambda)**2 + (e**2 - 1)**2)*sin(lambda) + (1 - e**2)*(-e**2*cos(lambda)**2 + 1)*atanh(e*sin(lambda)/sqrt(-e**2*(e**2 - 2)*sin(lambda)**2 + (e**2 - 1)**2)))/(e*(-e**2*cos(lambda)**2 + 1))

In [16]:
sp.simplify(integral.subs(e, sp.sqrt(f*(2-f))))

(sqrt(f)*sqrt(2 - f)*sqrt(-f*(f - 2)*(f*(f - 2) + 2)*sin(lambda)**2 + (f*(f - 2) + 1)**2)*sin(lambda) + (f*(f - 2) + 1)*(-f*(f - 2)*sin(lambda)**2 + f*(f - 2) + 1)*atanh(sqrt(f)*sqrt(2 - f)*sin(lambda)/sqrt(-f*(f - 2)*(f*(f - 2) + 2)*sin(lambda)**2 + (f*(f - 2) + 1)**2)))/(sqrt(f)*sqrt(2 - f)*(-f*(f - 2)*sin(lambda)**2 + f*(f - 2) + 1))

In [17]:
sp.simplify(integral.subs(lam, sp.pi/2))

-e*atanh(e) + 1 + atanh(e)/e

In [18]:
sp.simplify(integral.subs(lam, -sp.pi/2))

e*atanh(e) - 1 - atanh(e)/e

In [19]:
sp.simplify(integral.subs(lam, sp.pi/2) - integral.subs(lam, -sp.pi/2))

-2*e*atanh(e) + 2 + 2*atanh(e)/e

In [21]:
f = 1/297
e = np.sqrt(f * (2 - f))

1 + (1/e - e) * np.arctanh(e)

np.float64(1.995512176652856)

In [24]:
lats = np.array([0, np.pi/6, np.pi/4, np.pi/3, np.pi/2])
e2 = e**2

sqrt_corrections = np.sqrt(1 + (e2 * np.sin(lats) * np.cos(lats) / (e2 * (np.cos(lats))**2 - 1))**2)
sqrt_corrections

array([1.        , 1.00000428, 1.00000569, 1.00000425, 1.        ])